#Import

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col, length

In [0]:
RENAME_MAP = {
    "cst_id" : "customer_id",
    "cst_key" : "customer_key",
    "cst_firstname" : "first_name",
    "cst_lastname" : "last_name",
    "cst_marital_status" : "marital_status",
    "cst_gndr" : "gender",
    "cst_create_date" : "create_date"
}

#Reading From Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

#Data Transfromations

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))

In [0]:
df.select("cst_marital_status").distinct().display()

In [0]:
df.select("cst_gndr").distinct().display()

##Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(col("cst_marital_status")) == "S","Single")
         .when(F.upper(col("cst_marital_status")) == "M","Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(col("cst_gndr")) == "M","Male")
         .when(F.upper(col("cst_gndr")) == "F","Female")
         .otherwise("n/a")
    )
)

##Filter

In [0]:
df = df.filter(length(col("cst_key")) == 10)

#Ranameing The Columns

In [0]:
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

#Write Into Silver Table

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver.crm_customers")


In [0]:
%sql
SELECT * FROM silver.crm_customers WHERE LENGTH(customer_key) < 10